In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Clean start** — Run the cell below to remove any existing repo and clone fresh. Use this when you see "destination path already exists" or want a clean copy.

In [3]:
# Clean start: remove existing repo (if any), clone fresh, go to repo root
%cd /content
!rm -rf Fine-tunning-LLM
!git clone https://github.com/Jennt54321/Fine-tunning-LLM.git
%cd /content/Fine-tunning-LLM
!pwd

/content
Cloning into 'Fine-tunning-LLM'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 100 (delta 52), reused 87 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 2.18 MiB | 6.83 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/Fine-tunning-LLM
/content/Fine-tunning-LLM


In [7]:
# Use absolute path so we always land in repo root (avoids nesting if run multiple times)
%cd /content/Fine-tunning-LLM

/content/Fine-tunning-LLM


In [8]:
# Show the actual socratic_train_config.yaml in the cloned repo (Colab output)
config_path = "/content/Fine-tunning-LLM/socratic_train_config.yaml"
try:
    with open(config_path) as f:
        print(f"--- {config_path} ---\n")
        print(f.read())
except FileNotFoundError:
    print(f"Not found: {config_path} (run the clone + cd cells above first)")

--- /content/Fine-tunning-LLM/socratic_train_config.yaml ---

# 使用 GSM8K socratic 子集 + 研究用 prompt 做 SFT（Qwen2.5-3B-Instruct, M3 Mac）
# 執行: llamafactory-cli train socratic_train_config.yaml
# 需先跑: python prepare_socratic_dataset.py

model_name_or_path: Qwen/Qwen2.5-3B-Instruct
stage: sft
do_train: true
finetuning_type: lora
lora_target: all

dataset: gsm8k_socratic_train
dataset_dir: data/custom
template: qwen
cutoff_len: 1024  # 512 dropped ~8% of samples; 1024 keeps all (max ~760 tokens est.)
# max_samples: unset = use full training data (gsm8k_socratic_alpaca_train.jsonl)

output_dir: /content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct
overwrite_output_dir: false  # 從頭訓練，不從 checkpoint 恢復（換新 instruction 時必用）
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 0.0002
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.03

bf16: false
fp16: true
logging_steps: 5
save_strategy: steps
save_steps: 100  # ~2 checkpoints per epoch with full 7k 

In [5]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 102.6 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.4 MB/s eta 0:00:00:00:010:01m


In [9]:
!git submodule update --init --recursive

Submodule 'LLaMA-Factory' (https://github.com/hiyouga/LLaMA-Factory.git) registered for path 'LLaMA-Factory'
Cloning into '/content/Fine-tunning-LLM/LLaMA-Factory'...
Submodule path 'LLaMA-Factory': checked out 'bf04ca6af8c82bed0f9562d04c32a3b5851eaa06'


In [10]:
%cd LLaMA-Factory

/content/Fine-tunning-LLM/LLaMA-Factory


In [11]:
!pip install -e .[torch,bitsandbytes]

Obtaining file:///content/Fine-tunning-LLM/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.5.dev0-py3-none-any.whl size=27165 sha256=c37bc28195c52b98b54ac1accebccad9f5898e9bdae0879f87f35836575345e7
  Stored in directory: /tmp/pip-ephem-wheel-cache-hcxh5j8x/wheels/6e/a6/d6/b1e95b6c2acdb6be7059c425e768d03ffc31fade243bd712ab
Successfully built llamafactory
  Attempting uninstall: llamafactory
    Found existing installation: llamafactory 0.9.5.dev0
    Uninstalling llamafactory-0.9.5.dev0:
      Successfully uninstalled llamafactory-0.9.5.dev0


In [12]:
%cd ..

/content/Fine-tunning-LLM


**Checkpoint saving:** Training will save checkpoints to Google Drive (`output_dir` in config). Ensure Drive is mounted (cell above) and create the output dir so checkpoints are written reliably.

In [13]:
# Ensure checkpoint output dir exists on Drive (from socratic_train_config.yaml)
import os
output_dir = "/content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct"
os.makedirs(output_dir, exist_ok=True)
print(f"Checkpoints will be saved to: {output_dir}")

Checkpoints will be saved to: /content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct


In [14]:
# Quick test: 100 samples + save every 10 steps so we get a checkpoint (full run uses save_steps=100)
# CLI overrides use key=value (no space). overwrite_output_dir=false so we can test resume next.
!llamafactory-cli train socratic_train_config.yaml max_samples=100

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-02-10 14:09:55] llamafactory.hparams.parser:459 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
config.json: 100% 661/661 [00:00<00:00, 2.59MB/s]
[INFO|configuration_utils.py:667] 2026-02-10 14:09:55,577 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e7

In [15]:
# Verify checkpoints were saved (run after training finishes)
import os
output_dir = "/content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct"
if os.path.isdir(output_dir):
    items = sorted(os.listdir(output_dir))
    print("Saved in output_dir:", items)
    for name in items:
        path = os.path.join(output_dir, name)
        if os.path.isdir(path):
            print(f"  {name}/ -> {len(os.listdir(path))} files")
else:
    print("Output dir not found. Check that Drive is mounted and training completed.")

Saved in output_dir: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'all_results.json', 'chat_template.jinja', 'checkpoint-13', 'tokenizer.json', 'tokenizer_config.json', 'train_results.json', 'trainer_log.jsonl', 'trainer_state.json', 'training_args.bin', 'training_loss.png']
  checkpoint-13/ -> 12 files


In [17]:
!llamafactory-cli train socratic_train_config.yaml

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-02-10 14:19:24] llamafactory.hparams.parser:143 >> Resuming training from /content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct/checkpoint-13.
[INFO|2026-02-10 14:19:24] llamafactory.hparams.parser:143 >> Change `output_dir` or use `overwrite_output_dir` to avoid.
[INFO|2026-02-10 14:19:24] llamafactory.hparams.parser:459 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:667] 2026-02-10 14:19:25,023 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json
[INFO|configuration_utils.py:739] 2026-02-10 14:19:25,026 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_to

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found
